# 03. 전처리

- 입력: `data/processed/seoul_bike_clean.csv` (01에서 저장)
- 하는 일: humidity 0 보정 → 운영하지 않은 시간 제외 → 날짜 변수 추출 → 범주형 인코딩 → train/validation/test 분할(70/15/15) → 표준화
- 출력: `data/processed/X_train.csv`, `X_val.csv`, `X_test.csv`, `y_train.csv`, `y_val.csv`, `y_test.csv` → 04, 05에서 사용

In [1]:
import sys
from pathlib import Path

# 프로젝트 루트(notebooks/의 상위 폴더)를 import 경로에 추가해서 src 패키지를 사용
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.append(str(ROOT))

import pandas as pd

from src.config import SEED, TARGET
from src.data import load_clean, save_splits

df = load_clean()
print(df.shape)

(8760, 14)


In [2]:
import numpy as np

# humidity == 0 인 17행은 같은 행의 기온·이슬점과 맞지 않음 (측정 오류로 추정)
# → 기온·이슬점으로 상대습도를 계산(Magnus 공식)해서 대체
def humidity_from_dew_point(temp, dew_point):
    a, b = 17.625, 243.04
    return 100 * np.exp(a * dew_point / (b + dew_point)) / np.exp(a * temp / (b + temp))

zero_humidity = df['humidity'] == 0
estimated = humidity_from_dew_point(df.loc[zero_humidity, 'temperature'], df.loc[zero_humidity, 'dew_point_temp'])
df.loc[zero_humidity, 'humidity'] = estimated.round().astype(int)

print('보정한 행 수:', zero_humidity.sum())
print('보정 후 humidity == 0 행 수:', (df['humidity'] == 0).sum())
df.loc[zero_humidity, ['date', 'hour', 'temperature', 'dew_point_temp', 'humidity']]

보정한 행 수: 17
보정 후 humidity == 0 행 수: 0


,date,hour,temperature,dew_point_temp,humidity
4063,2018-05-19,7,11.4,4.5,63
4106,2018-05-21,2,13.9,-2.1,33
4107,2018-05-21,3,13.0,-2.5,34
4108,2018-05-21,4,12.4,-3.4,33
4109,2018-05-21,5,11.9,-2.7,36
4110,2018-05-21,6,11.4,-2.0,39
4111,2018-05-21,7,12.3,-2.7,35
4131,2018-05-22,3,15.5,10.4,72
4132,2018-05-22,4,15.6,10.5,72
4133,2018-05-22,5,15.7,10.6,72


In [3]:
# 운영하지 않은 시간(functioning_day == 'No', 295행)은 대여 수가 전부 0 → 수요가 아니므로 제외
# 제외 후에는 값이 'Yes' 하나뿐이라 컬럼도 삭제
data = df[df['functioning_day'] == 'Yes'].drop(columns='functioning_day').reset_index(drop=True)
print(df.shape, '->', data.shape)

(8760, 14) -> (8465, 13)


In [4]:
# 날짜에서 요일(0=월 ~ 6=일), 월, 주말 여부 추출 후 date 삭제
data['day_of_week'] = data['date'].dt.dayofweek
data['month'] = data['date'].dt.month
data['is_weekend'] = (data['day_of_week'] >= 5).astype(int)
data = data.drop(columns='date')
data[['day_of_week', 'month', 'is_weekend']].head()

C:\Users\JuHong\AppData\Local\Temp\ipykernel_2828\2942704856.py:2: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  data['day_of_week'] = data['date'].dt.dayofweek


,day_of_week,month,is_weekend
0,4,12,0
1,4,12,0
2,4,12,0
3,4,12,0
4,4,12,0


In [5]:
# 범주형 인코딩: seasons는 원-핫, holiday는 값이 2개라 0/1 한 컬럼으로 변환
data = pd.get_dummies(data, columns=['seasons'], dtype=int)
data['holiday'] = (data['holiday'] == 'Holiday').astype(int)
print(data.shape)
data.dtypes

(8465, 18)


rented_bike_count      int64
hour                   int64
temperature          float64
humidity               int64
wind_speed           float64
visibility             int64
dew_point_temp       float64
solar_radiation      float64
rainfall             float64
snowfall             float64
holiday                int64
day_of_week            int32
month                  int32
is_weekend             int64
seasons_Autumn         int64
seasons_Spring         int64
seasons_Summer         int64
seasons_Winter         int64
dtype: object

In [6]:
from sklearn.model_selection import train_test_split

X = data.drop(columns=TARGET)
y = data[TARGET]

# 무작위 분할: train 70 / validation 15 / test 15
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=SEED)

for name, part in [('train', X_train), ('validation', X_val), ('test', X_test)]:
    print(f'{name:10s} {len(part):5d}행 ({len(part) / len(X):.1%})')

train       5925행 (70.0%)
validation  1270행 (15.0%)
test        1270행 (15.0%)


In [7]:
from sklearn.preprocessing import StandardScaler

# 연속형·순서형 변수만 표준화 (0/1 컬럼은 그대로)
# 데이터 누수 방지: train으로만 fit, validation/test는 transform만
binary_cols = ['holiday', 'is_weekend'] + [c for c in X.columns if c.startswith('seasons_')]
scale_cols = [c for c in X.columns if c not in binary_cols]

scaler = StandardScaler()
X_train = X_train.copy()
X_val = X_val.copy()
X_test = X_test.copy()
X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_val[scale_cols] = scaler.transform(X_val[scale_cols])
X_test[scale_cols] = scaler.transform(X_test[scale_cols])

print('표준화한 컬럼:', scale_cols)
print('그대로 둔 컬럼:', binary_cols)
X_train[scale_cols].agg(['mean', 'std']).round(3)

표준화한 컬럼: ['hour', 'temperature', 'humidity', 'wind_speed', 'visibility', 'dew_point_temp', 'solar_radiation', 'rainfall', 'snowfall', 'day_of_week', 'month']
그대로 둔 컬럼: ['holiday', 'is_weekend', 'seasons_Autumn', 'seasons_Spring', 'seasons_Summer', 'seasons_Winter']


,hour,temperature,humidity,wind_speed,visibility,dew_point_temp,solar_radiation,rainfall,snowfall,day_of_week,month
mean,0.0,0.0,-0.0,0.0,-0.0,-0.0,0.0,0.0,0.0,-0.0,0.0
std,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


In [8]:
# 분할·표준화된 데이터 저장 (04, 05에서 사용)
save_splits(X_train, X_val, X_test, y_train, y_val, y_test)